# Tiempo hasta una segunda compra: preparación

Consulta la [metodología](../docs/metodologia.md) y ejecuta las etapas en orden. Los comentarios se calculan con los resultados guardados.

In [1]:
from pathlib import Path
import sys
here = Path.cwd().resolve()
def _buscar_proyecto(paquete):
    """Encuentra el paquete propio desde el proyecto, tema o portafolio."""
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (base / "src" / paquete).is_dir():
            return base
        for container in (base, base / "Analisis"):
            for pattern in (f"*/src/{paquete}", f"*/*/src/{paquete}"):
                for source in sorted(container.glob(pattern)):
                    parts = source.relative_to(container).parts
                    if "back" in parts or any(p.startswith(".") for p in parts):
                        continue
                    candidate = source.parent.parent
                    if (candidate / "pyproject.toml").is_file():
                        return candidate
    return None

PROJECT_ROOT = _buscar_proyecto("customer_return_survival")
if PROJECT_ROOT is None:
    raise RuntimeError("Abre este notebook desde su proyecto independiente.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from customer_return_survival.alerts import configurar_notebook, mostrar_alertas
from IPython.display import display, Markdown, Image
configurar_notebook()


In [2]:
from customer_return_survival.analysis import run_stage
import pandas as pd
metrics,comment=run_stage(PROJECT_ROOT,"preparacion")
display(Markdown(comment))
scalars={k:v for k,v in metrics.items() if not isinstance(v,(dict,list))}
if scalars:display(pd.DataFrame(list(scalars.items()),columns=["Indicador","Valor"]))
if "candidatos" in metrics:display(pd.DataFrame(metrics["candidatos"]))
for filename in ["eda.png","evaluacion.png"]:
    path=PROJECT_ROOT/"reports/preparacion"/filename
    if path.exists():display(Image(filename=str(path)))
for filename in ["estimaciones.csv","metricas_test.csv","horas_por_empleado.csv", "estabilidad_semillas.csv", "centralidad.csv"]:
    path=PROJECT_ROOT/"reports/preparacion"/filename
    if path.exists():display(pd.read_csv(path).head(20))

# Tiempo hasta una segunda compra: preparacion

Se construye una cohorte de **4338 clientes**: **1927 segundas ocasiones de compra** y **2411 casos censurados**. Seguimiento hasta 90 días o cierre del registro. Compras al mismo instante se agrupan; se excluyen 0 casos sin seguimiento positivo. Features derivadas exclusivamente de la primera ocasión observada. Particiones por cliente 60/20/20, aleatorias y estratificadas por evento; no constituyen una validación temporal.


,Indicador,Valor
0,cierre_observacion,2011-12-09T12:50:00
1,filas_originales,541909
2,filas_compras_positivas_identificadas,397884
3,ocasiones_compra,18440
4,clientes_sin_seguimiento_excluidos,0
5,fuente_sha256,f5385cbb54bbebf7196389109c6b0621faab0c304e3702...
6,clientes,4338
7,eventos,1927
8,censurados,2411
9,horizonte_dias,90


In [3]:
mostrar_alertas(PROJECT_ROOT / "reports/alertas/01_preparacion_datos.json")
from customer_return_survival.informe import generar_informe
generar_informe(PROJECT_ROOT)
display(Markdown("[Abrir informe integral](../reports/informe_final/informe_final.md)"))

[Abrir informe integral](../reports/informe_final/informe_final.md)